# 04. Поведенческая сегментация

> **Краткое резюме**: Кластеризуем клиентов по поведенческому профилю (обороты, контрагенты, направленность, динамика). Пересечение с модельным сегментом показывает, насколько поведение совпадает с формальной классификацией. Результат — каждый клиент получает `behavioral_segment`.

**Предпосылки**: `03_feature_engineering.ipynb` выполнен.

---

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

OUTPUT_DIR = os.path.abspath('./data')

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

K_MIN = 3       # Минимальное число кластеров
K_MAX = 10      # Максимальное
K_OVERRIDE = None  # Задать вручную (None = автоподбор по silhouette)

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, 'feature_matrix.parquet'))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, 'full_client_data.parquet'))

with open(os.path.join(OUTPUT_DIR, 'feature_meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

print(f'Feature matrix: {X.shape}')
print(f'Full data: {full_df.shape}')

---
## 2. Подбор оптимального k

Перебираем k от K_MIN до K_MAX, выбираем лучший по **silhouette score** (чем выше — тем лучше разделение).

In [ ]:
X_vals = X.values

if K_OVERRIDE is not None:
    best_k = K_OVERRIDE
    print(f'Using override k={best_k}')
else:
    max_k = min(K_MAX, len(X) - 1)
    results = []
    for k in range(K_MIN, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_vals)
        if len(set(labels)) < 2:
            continue
        sil = silhouette_score(X_vals, labels, sample_size=min(10000, len(X)))
        inertia = km.inertia_
        results.append({'k': k, 'silhouette': sil, 'inertia': inertia})
        print(f'  k={k}: silhouette={sil:.4f}, inertia={inertia:.0f}')

    res_df = pd.DataFrame(results)
    best_k = int(res_df.loc[res_df['silhouette'].idxmax(), 'k'])
    print(f'\nBest k={best_k} (silhouette={res_df["silhouette"].max():.4f})')

In [ ]:
# Визуализация подбора k
if K_OVERRIDE is None and len(res_df) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(res_df['k'], res_df['silhouette'], 'o-', linewidth=2)
    axes[0].axvline(best_k, color='red', linestyle='--', label=f'best k={best_k}')
    axes[0].set_xlabel('k')
    axes[0].set_ylabel('Silhouette Score')
    axes[0].set_title('Silhouette Score vs k')
    axes[0].legend()

    axes[1].plot(res_df['k'], res_df['inertia'], 'o-', linewidth=2, color='orange')
    axes[1].axvline(best_k, color='red', linestyle='--', label=f'best k={best_k}')
    axes[1].set_xlabel('k')
    axes[1].set_ylabel('Inertia')
    axes[1].set_title('Elbow Plot')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

---
## 3. Финальная кластеризация

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
full_df['behavioral_segment'] = km_final.fit_predict(X_vals)

print(f'Segments assigned (k={best_k}):')
print(full_df['behavioral_segment'].value_counts().sort_index())

---
## 4. Профили сегментов

Средние значения ключевых поведенческих метрик по сегментам.

In [ ]:
PROFILE_COLS = [
    'total_amount', 'tx_count', 'unique_counterparties',
    'direction_ratio', 'active_months', 'amount_growth', 'cp_growth',
]
# Добавляем опциональные
for col in ['avg_balance', 'product_count', 'avg_monthly_amount', 'tx_amount_cv']:
    if col in full_df.columns:
        PROFILE_COLS.append(col)

segment_profiles = full_df.groupby('behavioral_segment')[PROFILE_COLS].mean()
print('Segment profiles (mean):')
display(segment_profiles.round(2))

In [ ]:
# Размер сегментов
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = full_df['behavioral_segment'].value_counts().sort_index()
axes[0].bar(seg_counts.index.astype(str), seg_counts.values, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Count')
axes[0].set_title('Segment Sizes')

# Средний оборот
if 'avg_monthly_amount' in segment_profiles.columns:
    plot_col = 'avg_monthly_amount'
else:
    plot_col = 'total_amount'
axes[1].bar(segment_profiles.index.astype(str),
            segment_profiles[plot_col],
            edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel(plot_col)
axes[1].set_title(f'Mean {plot_col} by Segment')

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart
norm_profiles = pd.DataFrame(
    MinMaxScaler().fit_transform(segment_profiles),
    index=segment_profiles.index,
    columns=segment_profiles.columns,
)

labels = [c[:12] for c in norm_profiles.columns]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for i, (seg_id, row) in enumerate(norm_profiles.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals, 'o-', label=f'Seg {seg_id}',
            color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, vals, alpha=0.08, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_title('Segment Profiles (normalized)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0))
plt.tight_layout()
plt.show()

---
## 5. Пересечение с модельным сегментом

Насколько поведенческая сегментация совпадает с формальным модельным сегментом?

In [ ]:
if 'srvpackagesegment_ccode' in full_df.columns:
    cross = pd.crosstab(
        full_df['srvpackagesegment_ccode'].fillna('N/A'),
        full_df['behavioral_segment'],
        margins=True, margins_name='Total'
    )
    print('Model Segment x Behavioral Segment:')
    display(cross)

    # Heatmap
    cross_no_total = cross.drop('Total').drop('Total', axis=1)
    fig, ax = plt.subplots(figsize=(10, max(6, len(cross_no_total) * 0.5)))
    im = ax.imshow(cross_no_total.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(cross_no_total.shape[1]))
    ax.set_xticklabels(cross_no_total.columns, fontsize=10)
    ax.set_yticks(range(cross_no_total.shape[0]))
    ax.set_yticklabels(cross_no_total.index, fontsize=9)
    ax.set_xlabel('Behavioral Segment')
    ax.set_ylabel('Model Segment')
    ax.set_title('Model Segment x Behavioral Segment')
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()
else:
    print('Model segment not available.')

In [ ]:
# Пересечение с ОКВЭД
if 'sparkokved_ccode' in full_df.columns:
    okved_cross = pd.crosstab(
        full_df['sparkokved_ccode'].fillna('N/A'),
        full_df['behavioral_segment'],
        margins=True, margins_name='Total'
    )
    print('Top-15 OKVED x Behavioral Segment:')
    top_okveds = okved_cross.drop('Total')['Total'].nlargest(15).index
    display(okved_cross.loc[list(top_okveds) + ['Total']])

---
## 6. Сохранение

In [ ]:
seg_path = os.path.join(OUTPUT_DIR, 'segments.parquet')
full_df.to_parquet(seg_path)
print(f'Segments saved: {seg_path}')

# Сохраняем модель KMeans
model_path = os.path.join(OUTPUT_DIR, 'kmeans_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(km_final, f)
print(f'KMeans model saved: {model_path}')

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **K-Means** | Алгоритм кластеризации: разбивает данные на k групп по минимуму расстояний до центроидов |
| **Silhouette Score** | Качество кластеризации [-1, 1]: выше = лучше разделение |
| **Inertia (Elbow)** | Сумма расстояний до центроидов; «локоть» на графике = хорошее k |
| **behavioral_segment** | Номер кластера: группа клиентов с похожим поведенческим профилем |
| **Cross-tabulation** | Таблица пересечения двух категориальных переменных |

---

**Следующий шаг**: `05_lookalike.ipynb`.